# LoRA 微调

全量微调一个7B的模型需要56GB的显存，大多数公司都没有。LoRA让你用6GB就能做微调，只需要训练原本1%的参数量。这不是一种妥协，大多数任务上它与全量微调质量相近，所有的开源微调生态都依靠这个技巧。

## 问题描述

全量微调的问题：
1. 贵。
2. 修改所有的参数可能导致灾难性遗忘。

你需要一个能够训练较少参数，使用较低内存，并且不损害模型原油知识的方法。

## 基本概念

### LoRA

直觉是：微调过程中更新的参数有更小的内置秩。
```mermaid
flowchart LR
A[输入X] --> B[冻结模型参数 X x R] --> C[合并] --> D[输出]
A-->E[低秩矩阵X x R] --> F[低秩矩阵 R x X] --> C
```

初始化的时候，第一个低秩矩阵用随机高斯分布，第二个直接初始化成0。代表LoRA的贡献从0开始，也就是说模型从它的原始行为逐渐开始适应。

### 缩放因子

控制LoRA 两个低秩矩阵相乘后得到矩阵的缩放矩阵。是一个超参数，通常取2。

取1的话保守，但是稳定。更高的缩放因子意味着每步更大的更新，加速收敛，但也可能导致不稳定。

### 在哪些地方应用LoRA

一个Transformer有很多线性层，你不需要为所有的层都添加LoRA。
|层|微调参数（7B）|质量|
|---|---|---|
|q_proj|4.7M|Good|
|q_proj + v_proj|9.4M|Better|
|q_proj + k_proj + v_proj + o_proj|18.9M|Best for attention|
|All|37.7M|收益微薄，内存更多|

甜点区在q_proj + v_proj。控制模型应该关注什么，抽取什么。

### 秩R的选择

R=4，简单的分类任务，情感任务。

R=8、R=16，实践中最常见的选择。QA、摘要、遵循指令等任务。

R=32，复杂的代码生成、推理

R>64，收益不明显，而且开始失去LoRA的内存优势。

### QLoRA

在量化后的模型上挂载LoRA。

### 关于适配器

训练完成后，你有两个东西：冻结的基础模型，以及一个小的LoRA适配器。你可以选择：
- 保持分离。  加载基础模型，在此之上加载LoRA适配器。可以在不同任务间切换LoRA适配器，意味着你可以使用一个基础模型，但是能够做不同任务。
- 永久合并它们。  计算权重并相加，然后得到一个新的模型，跟基础模型一样大，不需要管理适配器。

可以同时部署多个适配器，但是要考虑技巧：
- TIES-Merging。 修剪小幅度参数，解决符号冲突，然后合并，减少适配器之间的干扰。
- DARE。 合并前随机丢弃适配器参数，然后缩放。
- Task Arithmetic。 直接相加或相减

### 什么时候别用微调

微调是你的第三个选择，不是第一个。

能用提示词搞定的事，优先提示器。

搞不定的用RAG。比如模型需要感知特定信息，检索比微调更便宜，且更容易维护。

最后才是微调。像让模型学会以某种语气、思维链作答，这些纯靠提示词是做不到的。结构化输出、蒸馏、或者延迟非常重要提示词不能过长的场景。

# 开始编码

In [8]:
import torch
import torch.nn as nn
import math

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=8, alpha=16):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.A = nn.Linear(in_dim, rank, bias=False)
        self.B = nn.Linear(rank, out_dim, bias=False)

        nn.init.normal_(self.A.weight, std=1 / math.sqrt(rank))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return (x @ self.A @ self.B) * self.scaling

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank=8, alpha=16):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

        for param in self.linear.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.linear(x) + self.lora(x)

def inject_lora(model, target_modules, rank=8, alpha=16):
    for param in model.parameters():
        param.requires_grad = False

    lora_layers = {}
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear):
            if any(t in name for t in target_modules):
                parent_name = ".".join(name.split(".")[:-1])
                child_name = name.split(".")[-1]
                if parent_name:
                    parent = dict(model.named_modules())[parent_name]
                else:
                    parent = model

                lora_linear = LinearWithLoRA(module, rank, alpha)
                setattr(parent, child_name, lora_linear)
                lora_layers[name] = lora_linear

    return lora_layers  

In [9]:
class TestNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 1)
        
test_model = TestNeuralNetwork()
inject_lora(test_model, ["fc1", "fc2"])

print(test_model.fc1)
print(test_model.fc2)



LinearWithLoRA(
  (linear): Linear(in_features=10, out_features=20, bias=True)
  (lora): LoRALayer(
    (A): Linear(in_features=10, out_features=8, bias=False)
    (B): Linear(in_features=8, out_features=20, bias=False)
  )
)
LinearWithLoRA(
  (linear): Linear(in_features=20, out_features=1, bias=True)
  (lora): LoRALayer(
    (A): Linear(in_features=20, out_features=8, bias=False)
    (B): Linear(in_features=8, out_features=1, bias=False)
  )
)


In [10]:
def train_lora(model, data, epochs=5, lr=1e-3, batch_size=4):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr
    )

    criterion = nn.MSELoss()

    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        n_batches = 0
        indices = torch.randperm(len(data["inputs"]))  

        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i: i + batch_size]
            x = data["inputs"][batch_idx]
            y = data["targets"][batch_idx]

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / max(n_batches, 1)
        losses.append(avg_loss)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return losses

            

In [12]:
def save_lora_adapter(model, path):
    adapter_state = {}
    for name, module in model.named_modules():
        if isinstance(model, LoRALayer):
            adapter_state[f"{name}.A"] = module.A.data.clone()
            adapter_state[f"{name}.B"] = module.B.data.clone()
            adapter_state[f"{name}.rank"] = module.rank
            adapter_state[f"{name}.alpha"] = module.alpha

    torch.save(adapter_state, path)
    return len(adapter_state) // 4

def load_lora_adapter(model, path):
    adapter_state = torch.load(path, weights_only=False)
    for name, module in model.named_modules():
        if isinstance(module, LoRALayer):
            a_key = f"{name}.A"
            b_key = f"{name}.B"
            if a_key in adapter_state:
                module.A.data = adapter_state[a_key]
                module.B.data = adapter_state[b_key]


# 已有生态

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Fake Model"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()